In [8]:
# Loading libraries
from biolearn.data_library import DataLibrary
from biolearn.data_library import GeoData
import pyreadr
import pandas as pd
import polars as pl
import pyarrow
from biolearn.model_gallery import ModelGallery

In [12]:
# Load Pheno data
pheno = pd.read_csv('/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/00pheno_sub-08042025.txt', sep='\t')
pheno

,name,SAB,Sentrix_ID,Year,Ethnicity,Gender,Age,PMIhrs,pH,Cause_of_Death,...,Hannum,Horvath1,PhenoAge,epiTOC2,StochClock1,StochClock2,StochClock3,EAA_StochClock1,EAA_StochClock2,EAA_StochClock3
0,BRA_201465920012_R01C01,67917,201465920012_R01C01,2018,Hispanic,Male,44,32.67,5.91,acute carisoprodol toxicity,...,17.174089,39.309067,-22.791375,181.134859,21.845551,4.513590,38.723731,-3.166403,-0.996325,-5.125392
1,BRA_201465920012_R02C01,67927,201465920012_R02C01,2018,Black,Female,32,31.57,6.56,undetermined,...,8.056451,39.536681,-17.402457,62.776804,17.806449,7.576214,45.290771,-2.723828,9.413272,2.261846
2,BRA_201465920012_R03C01,67928,201465920012_R03C01,2018,Black,Male,53,28.65,6.79,acute toxicity of cocaine and ethanol,...,17.936860,46.639434,-20.579133,369.318790,29.467912,1.795859,39.731010,1.094699,-9.224286,-4.733263
3,BRA_201465920012_R04C01,67929,201465920012_R04C01,2018,White,Female,36,28.20,6.80,toxic effects of methadone,...,15.410075,34.854667,-20.439521,901.551712,16.311634,2.873425,45.256590,-5.712535,2.261493,1.954265
4,BRA_201465920012_R05C01,67930,201465920012_R05C01,2018,White,Male,48,31.30,6.44,GSW,...,26.177655,43.418776,-30.526159,946.823647,26.861259,3.694324,42.525289,0.355412,-4.264582,-1.597234
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,BRA_205096370104_R04C01,67984,205096370104_R04C01,2022,White,Male,37,11.73,6.04,diabetic ketoacidosis,...,22.506465,42.242812,-22.832184,-38.586821,22.991093,1.306424,45.815595,0.593450,0.082244,2.444921
110,BRA_205096370104_R05C01,79212,205096370104_R05C01,2022,Black,Female,65,19.25,NaN,Suicide due to toxic effects of cyclobenzaprin...,...,33.098651,57.880399,-9.150546,-60.968314,38.449260,19.871303,46.616532,5.594370,1.504184,1.332061
111,BRA_205096370104_R07C01,79215,205096370104_R07C01,2022,Hispanic,Male,44,56.92,NaN,Bilateral pulmonary thromboemboli due to deep ...,...,24.057696,45.311226,-20.754439,-41.454759,30.714112,1.355331,42.775063,5.702157,-4.154584,-1.074061
112,BRA_205134980005_R01C01,79238,205134980005_R01C01,2022,Black,Female,66,23.83,NaN,NaN,...,28.314170,57.367596,-18.303709,-12.229472,34.984527,15.498145,42.140199,1.756163,-3.481222,-3.212622


In [3]:
# Read beta file (with index)
# Load the beta matrix
beta = pl.read_csv(
    "/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/01beta_rounded-08042025.csv"
)

In [4]:
# Convert to pandas DataFrame
beta_pd = beta.to_pandas()

# Set the first column as index if it's sample IDs
if beta_pd.columns[0] == "":
    beta_pd.rename(columns={beta_pd.columns[0]: "SampleID"}, inplace=True)
beta_pd.set_index(beta_pd.columns[0], inplace=True)

# Now call GeoData.from_methylation_matrix with pandas DataFrame
data = GeoData.from_methylation_matrix(beta_pd)
data.dnam

,cg00000622,cg00001245,cg00001510,cg00001583,cg00001594,cg00001687,cg00001747,cg00001793,cg00002028,cg00002116,...,cg08434186,cg12607287,cg06672160,cg14551162,cg20056602,cg05207355,cg07775305,cg09802893,cg06284364,cg05688618
SampleID,,,,,,,,,,,,,,,,,,,,,
BRA,0.01043,0.010325,0.499719,0.039404,0.017061,0.781614,0.01514,0.821833,0.025632,0.007579,...,0.948754,0.691123,0.862088,0.545605,0.873447,0.846351,0.85757,0.627053,0.669825,0.533851


In [7]:
# 4. Transpose to have samples as rows
beta_pd = beta_pd.transpose()
# 5. Now you can pass it to GeoData
data = GeoData.from_methylation_matrix(beta_pd)
data.dnam

SampleID,BRA_201465920012_R01C01,BRA_201465920012_R02C01,BRA_201465920012_R03C01,BRA_201465920012_R04C01,BRA_201465920012_R05C01,BRA_201465920012_R06C01,BRA_201465920012_R08C01,BRA_201465920048_R01C01,BRA_201465920048_R02C01,BRA_201465920048_R03C01,...,BRA_205096370081_R07C01,BRA_205096370081_R08C01,BRA_205096370104_R01C01,BRA_205096370104_R02C01,BRA_205096370104_R03C01,BRA_205096370104_R04C01,BRA_205096370104_R05C01,BRA_205096370104_R07C01,BRA_205134980005_R01C01,BRA_205134980005_R02C01
cg00000029,0.456,0.453,0.459,0.344,0.436,0.593,0.222,0.551,0.471,0.459,...,0.364,0.519,0.539,0.454,0.449,0.467,0.470,0.465,0.505,0.489
cg00000109,0.770,0.833,0.745,0.630,0.711,0.805,0.888,0.674,0.643,0.791,...,0.786,0.814,0.780,0.822,0.765,0.780,0.782,0.814,0.826,0.782
cg00000155,0.931,0.929,0.920,0.894,0.926,0.899,0.945,0.915,0.670,0.934,...,0.942,0.939,0.935,0.927,0.942,0.933,0.941,0.943,0.375,0.954
cg00000158,0.944,0.934,0.943,0.934,0.925,0.930,0.958,0.900,0.910,0.943,...,0.940,0.929,0.953,0.949,0.946,0.946,0.942,0.963,0.956,0.937
cg00000165,0.081,0.046,0.027,0.096,0.040,0.061,0.057,0.088,0.041,0.059,...,0.060,0.050,0.077,0.048,0.031,0.060,0.059,0.060,0.076,0.052
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
rs9363764,0.956,0.014,0.541,0.630,0.033,0.612,0.572,0.917,0.058,0.530,...,0.010,0.460,0.492,0.014,0.457,0.956,0.011,0.439,0.013,0.010
rs939290,0.019,0.012,0.009,0.945,0.651,0.012,0.603,0.617,0.618,0.943,...,0.467,0.548,0.938,0.940,0.573,0.490,0.943,0.951,0.485,0.927
rs951295,0.982,0.533,0.982,0.586,0.525,0.979,0.601,0.584,0.980,0.984,...,0.554,0.986,0.527,0.553,0.980,0.519,0.982,0.973,0.540,0.559
rs966367,0.427,0.449,0.938,0.668,0.633,0.516,0.430,0.594,0.668,0.470,...,0.396,0.954,0.355,0.393,0.960,0.358,0.952,0.009,0.366,0.966


In [10]:
# Predicting with Biolearn
gallery = ModelGallery()
#Note that by default clocks will impute missing data.
#To change this behavior set the imputation= parameter when getting the clock
CausAge = gallery.get("YingCausAge").predict(data)
DamAge = gallery.get("YingDamAge").predict(data)
AdaptAge = gallery.get("YingAdaptAge").predict(data)

In [18]:
# Combine into one DataFrame using pd.concat
merged_df = pd.concat([CausAge, DamAge, AdaptAge], axis=1)
# Rename the columns
merged_df.columns = ["CausAge", "DamAge", "AdaptAge"]
# Add index as a column named "name"
merged_df = merged_df.reset_index().rename(columns={"index": "name"})

In [19]:
# Combine with Pheno
pheno_com = pd.merge(pheno, merged_df, on='name')
pheno_com

,name,SAB,Sentrix_ID,Year,Ethnicity,Gender,Age,PMIhrs,pH,Cause_of_Death,...,epiTOC2,StochClock1,StochClock2,StochClock3,EAA_StochClock1,EAA_StochClock2,EAA_StochClock3,CausAge,DamAge,AdaptAge
0,BRA_201465920012_R01C01,67917,201465920012_R01C01,2018,Hispanic,Male,44,32.67,5.91,acute carisoprodol toxicity,...,181.134859,21.845551,4.513590,38.723731,-3.166403,-0.996325,-5.125392,61.659157,89.769912,-17.427697
1,BRA_201465920012_R02C01,67927,201465920012_R02C01,2018,Black,Female,32,31.57,6.56,undetermined,...,62.776804,17.806449,7.576214,45.290771,-2.723828,9.413272,2.261846,56.128294,98.530467,-25.842521
2,BRA_201465920012_R03C01,67928,201465920012_R03C01,2018,Black,Male,53,28.65,6.79,acute toxicity of cocaine and ethanol,...,369.318790,29.467912,1.795859,39.731010,1.094699,-9.224286,-4.733263,64.668921,90.596466,-3.609520
3,BRA_201465920012_R04C01,67929,201465920012_R04C01,2018,White,Female,36,28.20,6.80,toxic effects of methadone,...,901.551712,16.311634,2.873425,45.256590,-5.712535,2.261493,1.954265,57.276368,84.639845,1.553281
4,BRA_201465920012_R05C01,67930,201465920012_R05C01,2018,White,Male,48,31.30,6.44,GSW,...,946.823647,26.861259,3.694324,42.525289,0.355412,-4.264582,-1.597234,71.308171,88.779538,9.601219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,BRA_205096370104_R04C01,67984,205096370104_R04C01,2022,White,Male,37,11.73,6.04,diabetic ketoacidosis,...,-38.586821,22.991093,1.306424,45.815595,0.593450,0.082244,2.444921,64.455636,97.714300,-12.570473
110,BRA_205096370104_R05C01,79212,205096370104_R05C01,2022,Black,Female,65,19.25,NaN,Suicide due to toxic effects of cyclobenzaprin...,...,-60.968314,38.449260,19.871303,46.616532,5.594370,1.504184,1.332061,67.530403,95.870614,-9.721639
111,BRA_205096370104_R07C01,79215,205096370104_R07C01,2022,Hispanic,Male,44,56.92,NaN,Bilateral pulmonary thromboemboli due to deep ...,...,-41.454759,30.714112,1.355331,42.775063,5.702157,-4.154584,-1.074061,66.088791,93.359736,-17.125473
112,BRA_205134980005_R01C01,79238,205134980005_R01C01,2022,Black,Female,66,23.83,NaN,NaN,...,-12.229472,34.984527,15.498145,42.140199,1.756163,-3.481222,-3.212622,73.700413,100.408326,-15.757594


In [20]:
# Save
pheno_com.to_csv('/vast/palmer/scratch/montalvo-ortiz/jjm262/00tclocks/00uthealth/00databases/05epigenome/01pheno_uthealth_epiclocks-08042025.txt', sep='\t', index=False)